# event_json 기반 SHAP 공정 전이 불량 예측

1. `manufacturing_event_json.csv`의 `event_json` 파싱
2. `sourceTrace`로 원본 Bosch/Ford/열화상 라벨 복원
3. 동일 차량의 `PRESS → BODY → PAINT → ASSEMBLY` 이벤트 연결
4. 대상 공정 이전 JSON 특성으로 다음 공정 불량 학습
5. 후보 모델 비교 및 SHAP 기반 공정별 영향 분석
6. 공정 전이 위험 결과와 모델 산출물 저장

> CSV의 13번째 컬럼은 `is_sent`이며 불량 라벨이 아닙니다. 불량 정답은 JSON의
> `sourceTrace`가 가리키는 원본 데이터셋에서 학습 시점에만 복원합니다.
>
> 현재 이벤트는 서로 다른 공개 데이터셋을 조합한 합성 이벤트이므로 결과는
> **합성 데이터 기반 공정 전이 위험 예측**으로 해석합니다.


In [ ]:
# Colab 환경에서 필요한 패키지 설치
import importlib.util
import subprocess
import sys

def ensure_package(import_name: str, pip_name: str | None = None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name, "-q"])

ensure_package("lightgbm")
ensure_package("shap")
ensure_package("joblib")
ensure_package("optuna")
ensure_package("imblearn", "imbalanced-learn")


In [ ]:
# CPU 병렬 사용
import lightgbm as lgb

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    n_jobs=-1,
    random_state=42
)

> 필요한 추가 패키지(`optuna`, `imbalanced-learn`)는 위 설치 셀에서 자동 확인/설치합니다.


In [ ]:
from __future__ import annotations

import gc
import json
import pickle
import re
import shutil
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from optuna.samplers import TPESampler
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, IsolationForest, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (GroupShuffleSplit, ParameterSampler, StratifiedGroupKFold, train_test_split)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("pandas", pd.__version__)
print("lightgbm", lgb.__version__)
print("optuna", optuna.__version__)
print("shap", shap.__version__)


## 1. 데이터셋 및 산출물 경로 설정


In [ ]:
# Drive 마운트
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Colab이 아니므로 Google Drive mount를 건너뜁니다.")

PROJECT_ROOT = Path("/content/drive/MyDrive") if IN_COLAB else Path.cwd()
PROCESS_ROOT = (
    PROJECT_ROOT / "aims_dataset"
    if IN_COLAB
    else PROJECT_ROOT / "app" / "ml" / "datasets" / "process"
)
OUTPUT_DIR = Path("/content/defect_transfer_outputs") if IN_COLAB else Path("outputs/defect_transfer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_JSON_PATH = PROCESS_ROOT / "manufacturing_event_json.csv"
BOSCH_ROOT = PROCESS_ROOT / "bosch-production-line-performance"
THERMAL_ROOT = PROCESS_ROOT / "머신비전 AI 데이터셋 (열화상 기반 품질 검사 데이터)"
FORD_ROOT = PROCESS_ROOT / "Ford 엔진 진동 데이터셋"

NUMERIC_PATH = BOSCH_ROOT / "train_numeric.csv"
THERMAL_LEFT_LABEL_PATH = THERMAL_ROOT / "2nd_process_left_label.json"
THERMAL_RIGHT_LABEL_PATH = THERMAL_ROOT / "2nd_process_right_label.json"
FORD_TRAIN_PATH = FORD_ROOT / "FordA_TRAIN.txt"
FORD_TRAIN_ARFF_PATH = FORD_ROOT / "FordA_TRAIN.arff"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESS_ROOT:", PROCESS_ROOT)
print("EVENT_JSON_PATH:", EVENT_JSON_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. 대용량 event_json 로드 설정

CSV는 헤더가 없는 16컬럼 테이블이며 12번째 컬럼이 `event_json`, 13번째 컬럼이
`is_sent`입니다. 전체 파일을 메모리에 올리지 않고 chunk 단위로 읽습니다.


In [ ]:
# 학습 규모 설정
MAX_ROWS = 300_000
EVENT_CHUNK_SIZE = 20_000
ENABLE_HYPERPARAMETER_TUNING = True
CV_N_SPLITS = 5
CV_N_ITER = 15
ENABLE_OPTUNA_TUNING = True
CV_SAMPLE_SIZE = 30_000
FINAL_TRAIN_SAMPLE_SIZE = 120_000
RETRAIN_SELECTED_MODEL_ON_FULL_DATA = False
USE_SMOTE = False
SMOTE_SAMPLING_STRATEGY = 0.50
RECALL_PRIORITY_TARGET = 0.35
RECALL_PRIORITY_MIN_PRECISION = 0.02
THRESHOLD_BETA = 2.0

PROCESS_FLOW = ["PRESS", "BODY", "PAINT", "ASSEMBLY"]
TRANSITIONS = {
    "BODY": "PRESS",
    "PAINT": "BODY",
    "ASSEMBLY": "PAINT",
}
PROCESS_DISPLAY = {
    "PRESS": "프레스",
    "BODY": "차체",
    "PAINT": "도장",
    "ASSEMBLY": "의장",
    "FINAL_INSPECTION": "최종검사",
    "UNKNOWN": "미분류",
}

EVENT_COLUMNS = [
    "id", "event_id", "event_time", "car_master_id", "equipment_id",
    "process_code", "station_code", "equipment_code", "equipment_type",
    "equipment_status", "event_type", "event_json", "is_sent", "sent_at",
    "created_at", "updated_at",
]

# Colab Drive FUSE가 불안정할 수 있어 데이터셋을 런타임 캐시에 복사합니다.
LOCAL_DATA_CACHE = Path("/content/aims_dataset_cache") if IN_COLAB else None


def local_dataset_path(path: Path) -> Path:
    path = Path(path)
    if not IN_COLAB or LOCAL_DATA_CACHE is None:
        return path

    try:
        relative_path = path.relative_to(PROJECT_ROOT)
    except ValueError:
        relative_path = Path(path.name)

    cached_path = LOCAL_DATA_CACHE / relative_path
    if cached_path.exists():
        try:
            if cached_path.stat().st_size == path.stat().st_size:
                return cached_path
        except OSError:
            return cached_path

    cached_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"cache copy: {path} -> {cached_path}")
    shutil.copy2(path, cached_path)
    return cached_path


def assert_file(path: Path) -> None:
    if not path.exists():
        cached_path = local_dataset_path(path) if IN_COLAB else path
        if not cached_path.exists():
            raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")


def read_json_stable(path: Path):
    with open(local_dataset_path(path), "r", encoding="utf-8") as file:
        return json.load(file)


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="float")
    return df


In [ ]:
# 원본 데이터셋의 정답 라벨 로드
def load_ford_label_map() -> dict[int, int]:
    labels = {}
    path = local_dataset_path(FORD_TRAIN_PATH)
    if path.exists():
        with path.open(encoding="utf-8", errors="ignore") as file:
            for line in file:
                values = line.replace("\x00", " ").strip().split()
                if len(values) >= 501:
                    labels[len(labels) + 1] = int(float(values[0]))
                if len(labels) >= 4096:
                    break

    if labels:
        return labels

    # TXT가 손상되었거나 비어 있으면 ARFF의 마지막 class 값을 사용합니다.
    path = local_dataset_path(FORD_TRAIN_ARFF_PATH)
    in_data = False
    with path.open(encoding="utf-8", errors="ignore") as file:
        for line in file:
            value = line.strip()
            if not value or value.startswith("%"):
                continue
            if value.lower() == "@data":
                in_data = True
                continue
            if not in_data:
                continue
            parts = value.split(",")
            if len(parts) >= 501:
                labels[len(labels) + 1] = int(float(parts[-1]))
            if len(labels) >= 4096:
                break
    return labels


def load_vision_label_map() -> dict[tuple[str, int], int]:
    labels = {}
    for side, path in [
        ("LEFT", THERMAL_LEFT_LABEL_PATH),
        ("RIGHT", THERMAL_RIGHT_LABEL_PATH),
    ]:
        for row_id, label in enumerate(read_json_stable(path), start=1):
            labels[(side, row_id)] = int(float(label))
    return labels


def load_bosch_label_map() -> dict[int, int]:
    df = pd.read_csv(
        local_dataset_path(NUMERIC_PATH),
        usecols=["Id", "Response"],
        nrows=4096,
    )
    return dict(zip(df["Id"].astype(int), df["Response"].astype(int)))


for required_path in [
    EVENT_JSON_PATH,
    NUMERIC_PATH,
    THERMAL_LEFT_LABEL_PATH,
    THERMAL_RIGHT_LABEL_PATH,
]:
    assert_file(required_path)

ford_label_map = load_ford_label_map()
vision_label_map = load_vision_label_map()
bosch_label_map = load_bosch_label_map()

print("Ford labels:", len(ford_label_map), pd.Series(ford_label_map).value_counts().to_dict())
print("Vision labels:", len(vision_label_map), pd.Series(vision_label_map).value_counts().to_dict())
print("Bosch labels:", len(bosch_label_map), pd.Series(bosch_label_map).value_counts().to_dict())


## 3. event_json 특성 생성

동일 차량의 공정 이벤트를 순서대로 누적하고 대상 공정 직전까지의 JSON만 입력으로
사용합니다. `visionLabel`, `healthStatus`, 조립 오류 수처럼 원본 라벨에서 직접 생성된
필드는 입력에서 제외합니다.


In [ ]:
LEAKAGE_JSON_PATHS = {
    "equipmentStatus.healthStatus",
    "equipmentStatus.operationStatus",
    "processData.paint.visionLabel",
    "processData.paint.defectScore",
    "processData.paint.surfaceQualityScore",
    "processData.body.robotMotionStatus",
    "processData.assembly.actualSequence",
    "processData.assembly.missingPartCount",
    "processData.assembly.fasteningErrorCount",
    "processData.assembly.sequenceErrorCount",
}


def add_numeric_features(output: dict, prefix: str, values: dict) -> None:
    for key, value in values.items():
        feature_name = f"{prefix}{key}"
        if isinstance(value, bool):
            output[feature_name] = int(value)
        elif isinstance(value, (int, float)) and not isinstance(value, bool):
            output[feature_name] = float(value)


def extract_event_features(payload: dict) -> dict[str, float]:
    process_code = payload.get("location", {}).get("processCode", "UNKNOWN")
    prefix = f"{process_code}__"
    features = {}

    sensor = payload.get("sensor", {})
    for section in ["current", "vibration", "robotArmVibration", "thermal"]:
        add_numeric_features(features, f"{prefix}sensor_{section}_", sensor.get(section, {}))

    add_numeric_features(features, f"{prefix}metrics_", payload.get("processMetrics", {}))
    add_numeric_features(features, f"{prefix}manufacturing_", payload.get("manufacturing", {}))
    add_numeric_features(features, f"{prefix}product_", payload.get("product", {}))

    process_data = payload.get("processData", {})
    if process_code == "PRESS":
        add_numeric_features(features, f"{prefix}process_", process_data.get("press", {}))
    elif process_code == "BODY":
        add_numeric_features(
            features,
            f"{prefix}frequency_",
            process_data.get("body", {}).get("frequencyBands", {}),
        )
    elif process_code == "PAINT":
        paint = process_data.get("paint", {})
        for key in ["thermalStdTemp", "thicknessValue"]:
            if isinstance(paint.get(key), (int, float)):
                features[f"{prefix}process_{key}"] = float(paint[key])

    station_code = str(payload.get("location", {}).get("stationCode", ""))
    station_match = re.search(r"_(\d+)$", station_code)
    if station_match:
        features[f"{prefix}station_number"] = float(station_match.group(1))
    return features


def restore_target_label(payload: dict, target_process: str) -> tuple[int | None, str, int | None]:
    trace = payload.get("sourceTrace", {})
    if target_process == "BODY":
        row_id = int(trace.get("fordRowId", 0) or 0)
        raw_label = ford_label_map.get(row_id)
        return (int(raw_label < 0), "FordA.label", row_id) if raw_label is not None else (None, "FordA.label", row_id)

    if target_process == "PAINT":
        row_id = int(trace.get("machineVisionRowId", 0) or 0)
        side = str(payload.get("processData", {}).get("paint", {}).get("imagePosition", "")).upper()
        raw_label = vision_label_map.get((side, row_id))
        return (int(raw_label), f"thermal_{side.lower()}.label", row_id) if raw_label is not None else (None, "thermal.label", row_id)

    if target_process == "ASSEMBLY":
        row_id = int(trace.get("boschId", 0) or 0)
        raw_label = bosch_label_map.get(row_id)
        return (int(raw_label), "Bosch.Response", row_id) if raw_label is not None else (None, "Bosch.Response", row_id)

    return None, "unsupported", None


def build_transition_dataset(path: Path, max_rows: int | None) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_rows = []
    metadata_rows = []
    vehicle_state: dict[int, dict[str, dict[str, float]]] = {}
    parsed_rows = 0
    invalid_json_rows = 0
    missing_label_rows = 0

    reader = pd.read_csv(
        local_dataset_path(path),
        header=None,
        names=EVENT_COLUMNS,
        usecols=["event_id", "event_time", "car_master_id", "event_json"],
        chunksize=EVENT_CHUNK_SIZE,
        nrows=max_rows,
        dtype={"event_id": "string", "car_master_id": "int64", "event_json": "string"},
    )

    for chunk in reader:
        for row in chunk.itertuples(index=False):
            parsed_rows += 1
            try:
                payload = json.loads(row.event_json)
            except (TypeError, json.JSONDecodeError):
                invalid_json_rows += 1
                continue

            car_master_id = int(row.car_master_id)
            process_code = str(payload.get("location", {}).get("processCode", "UNKNOWN"))
            if process_code not in PROCESS_FLOW:
                continue

            if process_code == "PRESS" or car_master_id not in vehicle_state:
                vehicle_state[car_master_id] = {}
            history = vehicle_state[car_master_id]

            source_process = TRANSITIONS.get(process_code)
            if source_process and source_process in history:
                target_label, label_source, source_row_id = restore_target_label(payload, process_code)
                if target_label is None:
                    missing_label_rows += 1
                else:
                    features = {}
                    target_index = PROCESS_FLOW.index(process_code)
                    for history_process in PROCESS_FLOW[:target_index]:
                        features.update(history.get(history_process, {}))
                    for target_name in TRANSITIONS:
                        features[f"transition_to_{target_name}"] = int(process_code == target_name)
                    features["history_process_count"] = float(len(history))

                    feature_rows.append(features)
                    metadata_rows.append({
                        "sample_event_id": str(row.event_id),
                        "event_time": str(row.event_time),
                        "car_master_id": car_master_id,
                        "source_process_code": source_process,
                        "target_process_code": process_code,
                        "target_defect_label": int(target_label),
                        "label_source": label_source,
                        "label_source_row_id": source_row_id,
                    })

            history[process_code] = extract_event_features(payload)
            if process_code == "ASSEMBLY":
                vehicle_state.pop(car_master_id, None)

    feature_df = reduce_mem_usage(pd.DataFrame(feature_rows))
    metadata_df = pd.DataFrame(metadata_rows)
    print(
        f"event rows={parsed_rows:,}, transitions={len(feature_df):,}, "
        f"invalid_json={invalid_json_rows:,}, missing_label={missing_label_rows:,}"
    )
    return feature_df, metadata_df


features, transition_meta = build_transition_dataset(EVENT_JSON_PATH, MAX_ROWS)
if features.empty:
    raise ValueError("복원 가능한 공정 전이 학습 행이 없습니다.")

model_df = pd.concat(
    [transition_meta.reset_index(drop=True), features.reset_index(drop=True)],
    axis=1,
)
model_df = model_df.replace([np.inf, -np.inf], np.nan)

print("model_df:", model_df.shape)
display(model_df.head())


## 4. 라벨 복원 및 누수 검증

복원된 정답의 출처와 전이별 클래스 분포를 확인합니다. 대상 공정 JSON은 입력 특성에
포함하지 않으므로 예측 시점은 대상 공정 시작 전입니다.


In [ ]:
label_audit = transition_meta.groupby(
    ["source_process_code", "target_process_code", "label_source", "target_defect_label"]
).size().rename("rows").reset_index()
display(label_audit)

transition_summary = transition_meta.groupby(
    ["source_process_code", "target_process_code"]
).agg(
    rows=("target_defect_label", "size"),
    positive_rows=("target_defect_label", "sum"),
    positive_ratio=("target_defect_label", "mean"),
    vehicles=("car_master_id", "nunique"),
).reset_index()
display(transition_summary)


### 4.1 입력 특성 누수 점검


In [ ]:
def feature_process_prefix(feature: str) -> str | None:
    for process_code in PROCESS_FLOW:
        if feature.startswith(f"{process_code}__"):
            return process_code
    return None


invalid_target_features = []
for target_process in TRANSITIONS:
    target_rows = transition_meta["target_process_code"].eq(target_process)
    target_columns = [
        column for column in features.columns
        if column.startswith(f"{target_process}__")
    ]
    if target_columns:
        populated = features.loc[target_rows, target_columns].notna().any()
        invalid_target_features.extend(populated[populated].index.tolist())

if invalid_target_features:
    raise ValueError(
        f"대상 공정 feature가 입력에 포함되었습니다: "
        f"{sorted(set(invalid_target_features))[:10]}"
    )

for forbidden_name in [
    "visionLabel", "defectScore", "surfaceQualityScore", "robotMotionStatus",
    "healthStatus", "operationStatus", "actualSequence", "ErrorCount",
]:
    matched = [
        column for column in features.columns
        if forbidden_name.lower() in column.lower()
    ]
    if matched:
        raise ValueError(
            f"라벨 누수 의심 feature 발견({forbidden_name}): {matched[:10]}"
        )

print("대상 공정 및 라벨 파생 feature 누수 점검 통과")
print("feature count:", features.shape[1])


### 4.2 학습 데이터 품질 확인


In [ ]:
if transition_meta["target_defect_label"].nunique() < 2:
    raise ValueError("복원된 불량 라벨이 한 클래스만 포함되어 있습니다. MAX_ROWS를 늘려주세요.")

missing_ratio = features.isna().mean().sort_values(ascending=False)
display(missing_ratio.rename("missing_ratio").head(20).to_frame())
print("unique vehicles:", transition_meta["car_master_id"].nunique())
print("target distribution:", transition_meta["target_defect_label"].value_counts().to_dict())


## 5. 차량 단위 학습/검증 분리

같은 차량의 여러 공정 전이가 학습과 검증에 동시에 섞이지 않도록 `car_master_id`
기준 GroupShuffleSplit을 사용합니다.


In [ ]:
metadata_cols = [
    "sample_event_id", "event_time", "car_master_id", "source_process_code",
    "target_process_code", "target_defect_label", "label_source", "label_source_row_id",
]
feature_cols = [column for column in model_df.columns if column not in metadata_cols]
all_missing_cols = model_df[feature_cols].columns[model_df[feature_cols].isna().all()].tolist()
if all_missing_cols:
    model_df = model_df.drop(columns=all_missing_cols)
    feature_cols = [column for column in feature_cols if column not in all_missing_cols]

X = model_df[feature_cols]
y = model_df["target_defect_label"].astype(int)
ids = model_df["car_master_id"].astype(int)
groups = ids.copy()

group_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, valid_idx = next(group_split.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_valid = X.iloc[valid_idx]
y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]
id_train = ids.iloc[train_idx]
id_valid = ids.iloc[valid_idx]
groups_train = groups.iloc[train_idx]
groups_valid = groups.iloc[valid_idx]
meta_train = model_df.iloc[train_idx][metadata_cols]
meta_valid = model_df.iloc[valid_idx][metadata_cols]

if set(groups_train).intersection(set(groups_valid)):
    raise AssertionError("동일 차량이 train/validation에 동시에 포함되었습니다.")
if y_train.nunique() < 2 or y_valid.nunique() < 2:
    raise ValueError("차량 단위 분할 후 한 클래스만 남았습니다. MAX_ROWS를 늘려주세요.")

neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pos_weight = neg_count / max(pos_count, 1)

print("X_train:", X_train.shape, "X_valid:", X_valid.shape)
print("train vehicles:", groups_train.nunique(), "valid vehicles:", groups_valid.nunique())
print("positive ratio train:", y_train.mean(), "valid:", y_valid.mean())
print("scale_pos_weight:", round(scale_pos_weight, 2))


## 6. 전처리 학습 데이터셋 저장


In [ ]:
def save_training_dataframe(df: pd.DataFrame, output_dir: Path, stem: str) -> Path:
    parquet_path = output_dir / f"{stem}.parquet"
    csv_path = output_dir / f"{stem}.csv.gz"
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as exc:
        print(f"Parquet 저장 실패, csv.gz로 저장합니다: {exc}")
        df.to_csv(csv_path, index=False, compression="gzip")
        return csv_path


training_dataset_path = save_training_dataframe(
    model_df,
    OUTPUT_DIR,
    "defect_transfer_event_json_training_dataset",
)

split_rows = pd.concat([
    meta_train.assign(split="train"),
    meta_valid.assign(split="valid"),
]).sort_index()
split_ids_path = OUTPUT_DIR / "defect_transfer_train_valid_split_rows.csv"
split_rows.to_csv(split_ids_path, index=False)

preprocessing_metadata = {
    "event_json_path": str(EVENT_JSON_PATH),
    "event_json_column_index": 12,
    "is_sent_column_index": 13,
    "max_event_rows": int(MAX_ROWS),
    "event_chunk_size": int(EVENT_CHUNK_SIZE),
    "transition_rows": int(len(model_df)),
    "transition_summary": transition_summary.to_dict(orient="records"),
    "label_sources": {
        "BODY": "FordA label (-1 => defect)",
        "PAINT": "thermal vision label (1 => defect)",
        "ASSEMBLY": "Bosch Response (1 => defect)",
    },
    "excluded_leakage_json_paths": sorted(LEAKAGE_JSON_PATHS),
    "dropped_all_missing_columns": all_missing_cols,
    "feature_columns": feature_cols,
    "target_column": "target_defect_label",
    "split_strategy": "GroupShuffleSplit by car_master_id",
    "train_rows": int(len(X_train)),
    "valid_rows": int(len(X_valid)),
    "train_vehicles": int(groups_train.nunique()),
    "valid_vehicles": int(groups_valid.nunique()),
    "positive_ratio": float(y.mean()),
    "created_at": datetime.now(timezone.utc).isoformat(),
}
preprocessing_metadata_path = OUTPUT_DIR / "defect_transfer_preprocessing_metadata.json"
preprocessing_metadata_path.write_text(
    json.dumps(preprocessing_metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("전처리 학습 데이터셋:", training_dataset_path)
print("train/valid split rows:", split_ids_path)
print("전처리 metadata:", preprocessing_metadata_path)


## 7. 후보 모델 교차검증 및 하이퍼파라미터 튜닝

모든 후보 모델은 동일한 `event_json` 전이 학습 테이블을 사용합니다. 교차검증도
`car_master_id` 그룹을 유지하여 동일 차량의 행이 fold 사이에 섞이지 않게 합니다.
비교 지표는 PR-AUC, Recall, Precision, F1, ROC-AUC, Accuracy, False Positive Rate입니다.


In [ ]:
# 평가 함수
def find_recall_priority_threshold(
    y_true: pd.Series,
    proba: np.ndarray,
    *,
    min_recall: float = RECALL_PRIORITY_TARGET,
    min_precision: float = RECALL_PRIORITY_MIN_PRECISION,
    beta: float = THRESHOLD_BETA,
) -> float:
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    if len(thresholds) == 0:
        return 0.5

    precision = precision[:-1]
    recall = recall[:-1]
    thresholds = thresholds.astype(float)
    valid = (recall >= min_recall) & (precision >= min_precision)

    if valid.any():
        # Recall 목표를 만족하는 후보 중 Precision이 가장 좋은 threshold를 선택합니다.
        valid_idx = np.where(valid)[0]
        best_local_idx = valid_idx[int(np.nanargmax(precision[valid_idx]))]
        return float(thresholds[best_local_idx])

    beta_sq = beta ** 2
    f_beta = (1 + beta_sq) * precision * recall / np.maximum(beta_sq * precision + recall, 1e-12)
    return float(thresholds[int(np.nanargmax(f_beta))])


def find_best_threshold(y_true: pd.Series, proba: np.ndarray) -> float:
    return find_recall_priority_threshold(y_true, proba)


def compute_binary_metrics(y_true: pd.Series, proba: np.ndarray, threshold: float) -> dict:
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "pr_auc": float(average_precision_score(y_true, proba)),
        "false_positive_rate": float(fp / max(fp + tn, 1)),
        "predicted_positive_rate": float((tp + fp) / max(tp + fp + tn + fn, 1)),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


def predict_positive_proba(estimator, X_input: pd.DataFrame) -> np.ndarray:
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X_input)[:, 1]
    decision = estimator.decision_function(X_input)
    return 1 / (1 + np.exp(-decision))


def is_pipeline_estimator(estimator) -> bool:
    return isinstance(estimator, (Pipeline, ImbPipeline))


def get_final_estimator(estimator):
    return estimator.named_steps.get("model", estimator) if is_pipeline_estimator(estimator) else estimator


def make_lightgbm_pipeline() -> ImbPipeline:
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if USE_SMOTE:
        steps.append(("smote", SMOTE(
            sampling_strategy=SMOTE_SAMPLING_STRATEGY,
            random_state=RANDOM_STATE,
            k_neighbors=3,
        )))
    steps.append(("model", lgb.LGBMClassifier(
        objective="binary",
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        is_unbalance=True,
        verbosity=-1,
    )))
    return ImbPipeline(steps)


def make_candidate_models() -> dict:
    return {
        "LightGBM": {
            "estimator": make_lightgbm_pipeline(),
            "params": {
                "model__learning_rate": [0.015, 0.025, 0.04, 0.06],
                "model__num_leaves": [31, 63, 127],
                "model__max_depth": [-1, 6, 8, 12],
                "model__min_child_samples": [40, 80, 150, 250],
                "model__subsample": [0.75, 0.85, 1.0],
                "model__colsample_bytree": [0.65, 0.8, 0.95],
                "model__reg_lambda": [0.5, 1.0, 2.0, 5.0],
                "model__reg_alpha": [0.0, 0.1, 0.5],
            },
            "selection_reason": "대용량 tabular 제조 데이터에서 비선형 센서/시간 패턴을 빠르게 학습하는 기준 모델. Optuna와 SMOTE로 희소 불량 recall을 보강",
        },
        "RandomForest": {
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestClassifier(
                    n_estimators=160,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ]),
            "params": {
                "model__max_depth": [12, 16, 20, None],
                "model__min_samples_leaf": [1, 3, 5, 10],
                "model__max_features": ["sqrt", 0.35, 0.5],
            },
            "selection_reason": "bagging 기반 기준 모델로 과적합을 줄이고 안정적인 tree ensemble 성능 확인",
        },
        "ExtraTrees": {
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", ExtraTreesClassifier(
                    n_estimators=220,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ]),
            "params": {
                "model__max_depth": [12, 16, 20, None],
                "model__min_samples_leaf": [1, 3, 5, 10],
                "model__max_features": ["sqrt", 0.35, 0.5],
            },
            "selection_reason": "RandomForest보다 분할 무작위성이 커서 고차원 센서 feature의 강건성 비교",
        },
        "LogisticRegression": {
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(
                    class_weight="balanced",
                    max_iter=1500,
                    random_state=RANDOM_STATE,
                )),
            ]),
            "params": {
                "model__C": [0.03, 0.1, 0.3, 1.0, 3.0],
                "model__solver": ["lbfgs"],
            },
            "selection_reason": "복잡한 tree 모델 대비 선형 기준선으로 feature engineering 자체의 설명력 확인",
        },
    }


def suggest_lightgbm_params(trial: optuna.Trial) -> dict:
    return {
        "model__learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "model__num_leaves": trial.suggest_int("num_leaves", 24, 160),
        "model__max_depth": trial.suggest_categorical("max_depth", [-1, 5, 7, 9, 12]),
        "model__min_child_samples": trial.suggest_int("min_child_samples", 30, 300),
        "model__subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "model__colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
        "model__reg_lambda": trial.suggest_float("reg_lambda", 0.1, 8.0, log=True),
        "model__reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
    }


def evaluate_cv_trial(model_name: str, config: dict, params: dict, trial_no: int) -> tuple[dict, list[dict]]:
    fold_rows = []
    for fold_no, (train_idx, valid_idx) in enumerate(
    skf.split(X_cv_pool, y_cv_pool, groups=groups_cv_pool), start=1
):
        fold_started_at = datetime.now()
        X_fold_train = X_cv_pool.iloc[train_idx]
        y_fold_train = y_cv_pool.iloc[train_idx]
        X_fold_valid = X_cv_pool.iloc[valid_idx]
        y_fold_valid = y_cv_pool.iloc[valid_idx]

        cv_model = clone(config["estimator"])
        cv_model.set_params(**params)
        cv_model.fit(X_fold_train, y_fold_train)

        fold_proba = predict_positive_proba(cv_model, X_fold_valid)
        fold_threshold = find_best_threshold(y_fold_valid, fold_proba)
        fold_metrics = compute_binary_metrics(y_fold_valid, fold_proba, fold_threshold)
        fold_metrics.update({
            "model_name": model_name,
            "trial_no": trial_no,
            "fold_no": fold_no,
            "threshold": fold_threshold,
            **params,
        })
        fold_rows.append(fold_metrics)
        elapsed = (datetime.now() - fold_started_at).total_seconds()
        print(
            f"    fold {fold_no}/{effective_cv_splits} "
            f"PR-AUC={fold_metrics['pr_auc']:.5f} "
            f"Recall={fold_metrics['recall']:.5f} "
            f"F1={fold_metrics['f1']:.5f} "
            f"elapsed={elapsed:.1f}s"
        )

    fold_df = pd.DataFrame(fold_rows)
    summary = {
        "model_name": model_name,
        "trial_no": trial_no,
        "mean_pr_auc": float(fold_df["pr_auc"].mean()),
        "std_pr_auc": float(fold_df["pr_auc"].std(ddof=0)),
        "mean_roc_auc": float(fold_df["roc_auc"].mean()),
        "std_roc_auc": float(fold_df["roc_auc"].std(ddof=0)),
        "mean_accuracy": float(fold_df["accuracy"].mean()),
        "mean_false_positive_rate": float(fold_df["false_positive_rate"].mean()),
        "mean_precision": float(fold_df["precision"].mean()),
        "mean_recall": float(fold_df["recall"].mean()),
        "mean_f1": float(fold_df["f1"].mean()),
        "params": params,
        "selection_reason": config["selection_reason"],
        "tuning_method": "Optuna" if model_name == "LightGBM" and ENABLE_OPTUNA_TUNING else "ParameterSampler",
    }
    return summary, fold_rows


# CV 데이터셋 규모 관리: 차량 그룹을 유지한 채 표본 추출
if len(X_train) > CV_SAMPLE_SIZE:
    cv_fraction = min(0.99, CV_SAMPLE_SIZE / len(X_train))
    cv_sampler = GroupShuffleSplit(
        n_splits=1,
        train_size=cv_fraction,
        random_state=RANDOM_STATE,
    )
    cv_idx, _ = next(cv_sampler.split(X_train, y_train, groups=groups_train))
    X_cv_pool = X_train.iloc[cv_idx]
    y_cv_pool = y_train.iloc[cv_idx]
    groups_cv_pool = groups_train.iloc[cv_idx]
else:
    X_cv_pool = X_train
    y_cv_pool = y_train
    groups_cv_pool = groups_train

effective_cv_splits = min(
    CV_N_SPLITS,
    int(y_cv_pool.value_counts().min()),
    int(groups_cv_pool.nunique()),
)
if effective_cv_splits < 2:
    raise ValueError("교차검증을 수행하기에 소수 클래스 샘플이 부족합니다. MAX_ROWS 또는 CV_SAMPLE_SIZE를 늘려주세요.")

candidate_configs = make_candidate_models()
skf = StratifiedGroupKFold(n_splits=effective_cv_splits, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
cv_fold_rows = []
best_params_by_model = {}

print(f"CV rows: {len(X_cv_pool):,} / train rows: {len(X_train):,}")
print(f"CV splits: {effective_cv_splits}, tuning trials per model: {CV_N_ITER if ENABLE_HYPERPARAMETER_TUNING else 1}")
print(f"SMOTE enabled: {USE_SMOTE}, sampling_strategy={SMOTE_SAMPLING_STRATEGY}")
print(f"Threshold strategy: recall>={RECALL_PRIORITY_TARGET}, min_precision>={RECALL_PRIORITY_MIN_PRECISION}, beta={THRESHOLD_BETA}")

# 후보 모델별 교차검증
for model_name, config in candidate_configs.items():
    print(f"\n[{model_name}] CV start")
    model_started_at = datetime.now()

    if model_name == "LightGBM" and ENABLE_HYPERPARAMETER_TUNING and ENABLE_OPTUNA_TUNING:
        def objective(trial: optuna.Trial) -> float:
            params = suggest_lightgbm_params(trial)
            print(f"  optuna trial {trial.number + 1}/{CV_N_ITER} params={params}")
            summary, fold_rows = evaluate_cv_trial(model_name, config, params, trial.number + 1)
            cv_rows.append(summary)
            cv_fold_rows.extend(fold_rows)
            trial.set_user_attr("params", params)
            trial.set_user_attr("mean_recall", summary["mean_recall"])
            trial.set_user_attr("mean_f1", summary["mean_f1"])
            # PR-AUC를 우선하되 recall을 약하게 보상합니다.
            return summary["mean_pr_auc"] + 0.05 * summary["mean_recall"]

        study = optuna.create_study(
            direction="maximize",
            sampler=TPESampler(seed=RANDOM_STATE),
            study_name="event_json_transfer_lightgbm_recall_pr_auc",
        )
        study.optimize(objective, n_trials=CV_N_ITER, show_progress_bar=False)
        best_params_by_model[model_name] = study.best_trial.user_attrs["params"]
        print(f"[{model_name}] Optuna best value={study.best_value:.5f} params={best_params_by_model[model_name]}")
    else:
        param_candidates = list(ParameterSampler(
            config["params"],
            n_iter=CV_N_ITER,
            random_state=RANDOM_STATE,
        )) if ENABLE_HYPERPARAMETER_TUNING else [{}]
        print(f"[{model_name}] random search trials: {len(param_candidates)}")
        for trial_no, params in enumerate(param_candidates, start=1):
            print(f"  trial {trial_no}/{len(param_candidates)} params={params}")
            summary, fold_rows = evaluate_cv_trial(model_name, config, params, trial_no)
            cv_rows.append(summary)
            cv_fold_rows.extend(fold_rows)
            print(
                f"  -> {model_name} trial {trial_no} mean "
                f"PR-AUC={summary['mean_pr_auc']:.5f} "
                f"Recall={summary['mean_recall']:.5f} "
                f"F1={summary['mean_f1']:.5f} "
                f"FPR={summary['mean_false_positive_rate']:.5f}"
            )

    print(f"[{model_name}] CV done elapsed={(datetime.now() - model_started_at).total_seconds():.1f}s")

cv_results = pd.DataFrame(cv_rows).sort_values(
    ["mean_pr_auc", "mean_recall", "mean_f1", "mean_roc_auc", "mean_accuracy"],
    ascending=False,
).reset_index(drop=True)
cv_fold_results = pd.DataFrame(cv_fold_rows)

for model_name in candidate_configs:
    if model_name not in best_params_by_model:
        best_params_by_model[model_name] = cv_results[cv_results["model_name"].eq(model_name)].iloc[0]["params"]

best_model_name = str(cv_results.iloc[0]["model_name"])
best_params = best_params_by_model[best_model_name]
best_model_selection_reason = str(cv_results.iloc[0]["selection_reason"])

cv_results_path = OUTPUT_DIR / "defect_model_cv_results.csv"
cv_fold_results_path = OUTPUT_DIR / "defect_model_cv_fold_results.csv"
cv_results.to_csv(cv_results_path, index=False)
cv_fold_results.to_csv(cv_fold_results_path, index=False)

display(cv_results.head(20))
print("CV 결과 저장:", cv_results_path)
print("CV fold 결과 저장:", cv_fold_results_path)
print("CV best model:", best_model_name, best_params)


## 8. 후보 모델 최종 학습 및 성능 비교

교차검증에서 찾은 후보별 best parameter로 최종 학습하고 차량 단위 validation에서
다음 공정 불량 예측 성능을 비교합니다.


In [ ]:
trained_models = {}
model_valid_probabilities = {}
model_thresholds = {}
model_confusion_matrices = {}
model_eval_rows = []

# 최종 후보 비교용 학습 데이터셋 규모 관리
if len(X_train) > FINAL_TRAIN_SAMPLE_SIZE:
    X_final_train, _, y_final_train, _ = train_test_split(
        X_train,
        y_train,
        train_size=FINAL_TRAIN_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_train,
    )
else:
    X_final_train = X_train
    y_final_train = y_train

print(f"Final candidate training rows: {len(X_final_train):,} / train rows: {len(X_train):,}")

# 후보 모델 전체 학습
for model_name, config in candidate_configs.items():
    started_at = datetime.now()
    estimator = clone(config["estimator"])
    estimator.set_params(**best_params_by_model[model_name])
    print(f"[{model_name}] final fit start params={best_params_by_model[model_name]}")
    estimator.fit(X_final_train, y_final_train)

    proba = predict_positive_proba(estimator, X_valid)
    threshold = find_best_threshold(y_valid, proba)
    pred = (proba >= threshold).astype(int)
    cm_model = confusion_matrix(y_valid, pred, labels=[0, 1])
    metrics = compute_binary_metrics(y_valid, proba, threshold)

    trained_models[model_name] = estimator
    model_valid_probabilities[model_name] = proba
    model_thresholds[model_name] = threshold
    model_confusion_matrices[model_name] = cm_model

    model_eval_rows.append({
        "model_name": model_name,
        "threshold": threshold,
        "selection_reason": config["selection_reason"],
        **metrics,
    })
    print(
        f"[{model_name}] final fit done "
        f"PR-AUC={metrics['pr_auc']:.5f} "
        f"ROC-AUC={metrics['roc_auc']:.5f} "
        f"ACC={metrics['accuracy']:.5f} "
        f"FPR={metrics['false_positive_rate']:.5f} "
        f"elapsed={(datetime.now() - started_at).total_seconds():.1f}s"
    )

model_comparison = pd.DataFrame(model_eval_rows).sort_values(
    ["pr_auc", "recall", "f1", "roc_auc", "accuracy"],
    ascending=False,
).reset_index(drop=True)

selected_model_name = str(model_comparison.iloc[0]["model_name"])
model = trained_models[selected_model_name]
valid_proba = model_valid_probabilities[selected_model_name]
best_threshold = float(model_thresholds[selected_model_name])
valid_pred = (valid_proba >= best_threshold).astype(int)
cm = model_confusion_matrices[selected_model_name]

# 운영용 전체 재학습은 선택 모델 1개만 수행
if RETRAIN_SELECTED_MODEL_ON_FULL_DATA and len(X_final_train) < len(X_train):
    print(f"[{selected_model_name}] selected model full-data refit start rows={len(X_train):,}")
    refit_started_at = datetime.now()
    full_model = clone(candidate_configs[selected_model_name]["estimator"])
    full_model.set_params(**best_params_by_model[selected_model_name])
    full_model.fit(X_train, y_train)

    model = full_model
    trained_models[selected_model_name] = full_model
    valid_proba = predict_positive_proba(model, X_valid)
    best_threshold = find_best_threshold(y_valid, valid_proba)
    valid_pred = (valid_proba >= best_threshold).astype(int)
    cm = confusion_matrix(y_valid, valid_pred, labels=[0, 1])
    full_metrics = compute_binary_metrics(y_valid, valid_proba, best_threshold)
    model_comparison.loc[model_comparison["model_name"].eq(selected_model_name), list(full_metrics.keys())] = list(full_metrics.values())
    model_comparison.loc[model_comparison["model_name"].eq(selected_model_name), "threshold"] = best_threshold
    model_thresholds[selected_model_name] = best_threshold
    model_confusion_matrices[selected_model_name] = cm
    print(f"[{selected_model_name}] full-data refit done elapsed={(datetime.now() - refit_started_at).total_seconds():.1f}s")

roc_auc = float(model_comparison[model_comparison["model_name"].eq(selected_model_name)].iloc[0]["roc_auc"])
pr_auc = float(model_comparison[model_comparison["model_name"].eq(selected_model_name)].iloc[0]["pr_auc"])
accuracy = float(model_comparison[model_comparison["model_name"].eq(selected_model_name)].iloc[0]["accuracy"])
false_positive_rate = float(model_comparison[model_comparison["model_name"].eq(selected_model_name)].iloc[0]["false_positive_rate"])

# AI 관리 요약
baseline_row = model_comparison[model_comparison["model_name"].eq("LightGBM")]
baseline_pr_auc = float(baseline_row.iloc[0]["pr_auc"]) if not baseline_row.empty else np.nan
best_pr_auc = float(model_comparison[model_comparison["model_name"].eq(selected_model_name)].iloc[0]["pr_auc"])
performance_improvement = best_pr_auc - baseline_pr_auc if not np.isnan(baseline_pr_auc) else 0.0

ai_management_summary = {
    "모델_선정_근거": f"{selected_model_name} 모델이 validation PR-AUC/Recall/F1 기준 종합 순위 1위입니다. {candidate_configs[selected_model_name]['selection_reason']}",
    "데이터셋_규모_관리": {
        "max_event_rows": int(MAX_ROWS),
        "transition_rows": int(len(model_df)),
        "train_rows": int(len(X_train)),
        "candidate_train_rows": int(len(X_final_train)),
        "valid_rows": int(len(X_valid)),
        "feature_count": int(len(feature_cols)),
        "positive_ratio": float(y.mean()),
        "transition_summary": transition_summary.to_dict(orient="records"),
        "full_data_refit": bool(RETRAIN_SELECTED_MODEL_ON_FULL_DATA),
    },
    "테스트_케이스_수_관리": {
        "validation_total": int(len(y_valid)),
        "validation_normal": int((y_valid == 0).sum()),
        "validation_defect": int((y_valid == 1).sum()),
        "cv_splits": int(effective_cv_splits),
        "cv_sample_size": int(len(X_cv_pool)),
        "cv_trials_per_model": int(CV_N_ITER if ENABLE_HYPERPARAMETER_TUNING else 1),
        "lightgbm_tuning_method": "Optuna" if ENABLE_OPTUNA_TUNING else "ParameterSampler",
        "smote_enabled": bool(USE_SMOTE),
        "smote_sampling_strategy": float(SMOTE_SAMPLING_STRATEGY),
        "threshold_strategy": {
            "type": "recall_priority",
            "target_recall": float(RECALL_PRIORITY_TARGET),
            "min_precision": float(RECALL_PRIORITY_MIN_PRECISION),
            "beta": float(THRESHOLD_BETA),
        },
    },
    "정확도_Accuracy": accuracy,
    "오탐률_False_Positive_Rate": false_positive_rate,
    "모델_성능_개선_현황": {
        "baseline_model": "LightGBM",
        "baseline_pr_auc": baseline_pr_auc,
        "selected_model": selected_model_name,
        "selected_pr_auc": best_pr_auc,
        "pr_auc_improvement_over_lightgbm": performance_improvement,
    },
}

print("selected_model_name:", selected_model_name)
print("ROC-AUC:", round(roc_auc, 5))
print("PR-AUC:", round(pr_auc, 5))
print("Accuracy:", round(accuracy, 5))
print("False Positive Rate:", round(false_positive_rate, 5))
print("Recall:", round(float(model_comparison[model_comparison["model_name"].eq(selected_model_name)].iloc[0]["recall"]), 5))
display(model_comparison)
display(pd.DataFrame([ai_management_summary]))


## 9. 후보 모델별 Threshold 튜닝 및 Confusion Matrix

각 후보 모델은 validation set에서 recall 우선 threshold를 사용합니다.

기본 정책은 `RECALL_PRIORITY_TARGET` 이상을 만족하는 threshold 중 precision이 가장 높은 값을 선택하고, 목표 recall을 만족하는 후보가 없으면 F-beta(기본 beta=2) 기준으로 선택합니다. Accuracy는 클래스 불균형 때문에 참고 지표로만 해석합니다.


In [ ]:
print("== Selected Model ==")
print("model:", selected_model_name)
print("best_threshold:", round(best_threshold, 5))
print(classification_report(y_valid, valid_pred, digits=4, zero_division=0))

# 후보 모델별 Confusion Matrix
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for ax, (model_name, cm_model) in zip(axes, model_confusion_matrices.items()):
    ConfusionMatrixDisplay(cm_model, display_labels=["Normal", "Defect"]).plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
        colorbar=False,
    )
    row = model_comparison[model_comparison["model_name"].eq(model_name)].iloc[0]
    ax.set_title(
        f"{model_name}\n"
        f"Recall={row['recall']:.3f}, Precision={row['precision']:.3f}, F1={row['f1']:.3f}"
    )
plt.tight_layout()
plt.show()

# 비교 지표 테이블
display(model_comparison[[
    "model_name",
    "accuracy",
    "false_positive_rate",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "threshold",
]])


In [ ]:
# 모델 성능 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.countplot(x=y, ax=axes[0])
axes[0].set_title("Target Defect Distribution")
axes[0].set_xlabel("Target Defect Label")

sns.barplot(data=model_comparison, x="model_name", y="pr_auc", ax=axes[1], color="#4C78A8")
axes[1].set_title("Validation PR-AUC by Model")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

sns.barplot(data=model_comparison, x="model_name", y="false_positive_rate", ax=axes[2], color="#F58518")
axes[2].set_title("False Positive Rate by Model")
axes[2].set_xlabel("")
axes[2].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

# 선택 모델 ROC/PR 곡선
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_valid, valid_proba, ax=axes[0])
axes[0].set_title(f"{selected_model_name} ROC AUC={roc_auc:.4f}")
PrecisionRecallDisplay.from_predictions(y_valid, valid_proba, ax=axes[1])
axes[1].set_title(f"{selected_model_name} PR-AUC={pr_auc:.4f}")
plt.tight_layout()
plt.show()


def extract_feature_importance(estimator, feature_names: list[str], model_name: str) -> pd.DataFrame:
    target_estimator = estimator
    if is_pipeline_estimator(estimator):
        target_estimator = get_final_estimator(estimator)

    if hasattr(target_estimator, "booster_"):
        values = target_estimator.booster_.feature_importance(importance_type="gain")
        importance_type = "gain"
    elif hasattr(target_estimator, "feature_importances_"):
        values = target_estimator.feature_importances_
        importance_type = "feature_importances"
    elif hasattr(target_estimator, "coef_"):
        values = np.abs(target_estimator.coef_).ravel()
        importance_type = "abs_coef"
    else:
        values = np.zeros(len(feature_names))
        importance_type = "not_available"

    return pd.DataFrame({
        "model_name": model_name,
        "feature": feature_names,
        "importance": values,
        "importance_type": importance_type,
    }).sort_values("importance", ascending=False)


importance_frames = [
    extract_feature_importance(estimator, feature_cols, model_name)
    for model_name, estimator in trained_models.items()
]
all_model_importance = pd.concat(importance_frames, ignore_index=True)
importance_df = all_model_importance[all_model_importance["model_name"].eq(selected_model_name)].copy()
importance_df = importance_df.rename(columns={"importance": "importance_gain"})

plt.figure(figsize=(9, 7))
sns.barplot(data=importance_df.head(25), y="feature", x="importance_gain", color="#4C78A8")
plt.title(f"Top 25 Feature Importance - {selected_model_name}")
plt.xlabel("Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()

display(importance_df.head(30))


## 10. SHAP 기반 영향 공정 분석

선택된 모델의 SHAP 값을 JSON feature의 공정 prefix(`PRESS__`, `BODY__`, `PAINT__`)로
집계합니다. 이를 통해 다음 공정 불량 확률을 높인 이전 공정과 주요 센서값을 계산합니다.


In [ ]:
# SHAP 샘플링
SHAP_SAMPLE_SIZE = min(1000, len(X_valid))

if selected_model_name == "LogisticRegression":
    SHAP_SAMPLE_SIZE = min(500, len(X_valid))

X_shap = X_valid.sample(n=SHAP_SAMPLE_SIZE, random_state=RANDOM_STATE)
id_shap = id_valid.loc[X_shap.index]
y_shap = y_valid.loc[X_shap.index]
proba_shap = predict_positive_proba(model, X_shap)


def prepare_shap_model(estimator, X_sample: pd.DataFrame):
    if is_pipeline_estimator(estimator):
        final_estimator = get_final_estimator(estimator)
        transform_steps = estimator.steps[:-1]

        X_transformed = X_sample.copy()

        for _, step in transform_steps:
            if hasattr(step, "transform"):
                X_transformed = step.transform(X_transformed)

        X_transformed = pd.DataFrame(
            X_transformed,
            columns=X_sample.columns,
            index=X_sample.index
        )

        return final_estimator, X_transformed

    return estimator, X_sample


shap_model, X_shap_model = prepare_shap_model(model, X_shap)

# SHAP Explainer 생성
try:
    explainer = shap.TreeExplainer(shap_model)

    raw_shap_values = explainer.shap_values(
        X_shap_model,
        check_additivity=False
    )

    if isinstance(raw_shap_values, list):
        shap_matrix = raw_shap_values[1]
    elif getattr(raw_shap_values, "ndim", 0) == 3:
        shap_matrix = raw_shap_values[:, :, 1]
    else:
        shap_matrix = raw_shap_values

    expected_value = explainer.expected_value

    if isinstance(expected_value, list):
        base_value = expected_value[1]
    else:
        base_value = expected_value

except Exception as e:
    print("TreeExplainer 실패:", e)
    print("Permutation Explainer로 대체 실행합니다.")

    background = shap.sample(
        X_train,
        min(100, len(X_train)),
        random_state=RANDOM_STATE
    )

    explainer = shap.Explainer(
        lambda data: predict_positive_proba(
            model,
            pd.DataFrame(data, columns=X_train.columns)
        ),
        background
    )

    raw_explanation = explainer(
        X_shap,
        max_evals=2 * X_shap.shape[1] + 1
    )

    shap_matrix = raw_explanation.values
    base_value = raw_explanation.base_values
    X_shap_model = X_shap


# base_value 길이 보정
if np.isscalar(base_value):
    base_values = np.repeat(base_value, len(X_shap))
else:
    base_values = np.array(base_value)

    if base_values.ndim == 0:
        base_values = np.repeat(float(base_values), len(X_shap))
    elif len(base_values) != len(X_shap):
        base_values = np.repeat(float(np.ravel(base_values)[0]), len(X_shap))


# SHAP 객체 구성
shap_values = shap.Explanation(
    values=shap_matrix,
    base_values=base_values,
    data=X_shap_model.values,
    feature_names=X_shap.columns.tolist(),
)


print("selected_model_name:", selected_model_name)
print("X_shap:", X_shap.shape)
print("X_shap_model:", X_shap_model.shape)
print("shap_matrix:", shap_matrix.shape)
print("base_values:", np.array(base_values).shape)


In [ ]:
# SHAP Summary Plot
shap.summary_plot(shap_values, X_shap, max_display=25, show=True)

# SHAP 중요도 산출
shap_importance = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": np.abs(shap_matrix).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

display(shap_importance.head(30))


In [ ]:
# Feature 공정 매핑
def feature_to_process(feature: str) -> str | None:
    for process_code in PROCESS_FLOW:
        if feature.startswith(f"{process_code}__"):
            return process_code
    return None


def probability_to_risk_grade(prob: float) -> str:
    if prob >= 0.70:
        return "HIGH"
    if prob >= 0.40:
        return "MEDIUM"
    return "LOW"


feature_process = pd.Series(
    [feature_to_process(column) for column in X_shap.columns],
    index=X_shap.columns,
)
mapped_feature_mask = feature_process.isin(PROCESS_FLOW)
unmapped_features = feature_process[~mapped_feature_mask].index.tolist()
print(f"공정 영향도 집계 제외 전역 feature 수: {len(unmapped_features)}")
if unmapped_features:
    display(pd.DataFrame({"unmapped_feature": unmapped_features}).head(30))

process_rows = []
for process_code in PROCESS_FLOW:
    column_idx = np.where(feature_process.values == process_code)[0]
    process_rows.append({
        "process_code": process_code,
        "mean_abs_shap": float(np.abs(shap_matrix[:, column_idx]).mean()) if len(column_idx) else 0.0,
        "feature_count": int(len(column_idx)),
    })

process_importance = pd.DataFrame(process_rows).sort_values("mean_abs_shap", ascending=False)
display(process_importance)

plt.figure(figsize=(8, 4))
sns.barplot(data=process_importance, x="process_code", y="mean_abs_shap", color="#59A14F")
plt.title("SHAP Process Influence")
plt.xlabel("Source Process")
plt.ylabel("Mean |SHAP|")
plt.tight_layout()
plt.show()


## 11. SHAP 기반 공정 전이 위험 결과 생성

실제 학습 행의 원인·대상 공정 정보를 유지하면서 다음 공정 불량 확률, 가장 영향이 큰
JSON feature, 공정별 SHAP 영향 비율을 결과로 생성합니다.


In [ ]:
def build_transfer_predictions(
    metadata: pd.DataFrame,
    probabilities: np.ndarray,
    shap_matrix: np.ndarray,
    feature_names: list[str],
    feature_process: pd.Series,
) -> pd.DataFrame:
    records = []
    process_values = feature_process.values
    mapped_mask = feature_process.isin(PROCESS_FLOW).to_numpy()
    predicted_at = datetime.now(timezone.utc).isoformat()

    for row_idx, (_, metadata_row) in enumerate(metadata.iterrows()):
        probability = float(probabilities[row_idx])
        abs_values = np.abs(shap_matrix[row_idx])
        mapped_total = float(abs_values[mapped_mask].sum()) or 1.0

        process_scores = {}
        for process_code in PROCESS_FLOW:
            column_idx = np.where(process_values == process_code)[0]
            process_scores[process_code] = (
                float(abs_values[column_idx].sum() / mapped_total)
                if len(column_idx) else 0.0
            )

        source_process = str(metadata_row["source_process_code"])
        target_process = str(metadata_row["target_process_code"])
        available_idx = np.where(
            np.isin(process_values, PROCESS_FLOW[:PROCESS_FLOW.index(target_process)])
        )[0]
        top_feature_idx = (
            int(available_idx[np.argmax(abs_values[available_idx])])
            if len(available_idx)
            else int(abs_values.argmax())
        )

        records.append({
            "sample_event_id": metadata_row["sample_event_id"],
            "car_master_id": int(metadata_row["car_master_id"]),
            "source_process_code": source_process,
            "target_process_code": target_process,
            "target_defect_label": int(metadata_row["target_defect_label"]),
            "target_defect_probability": round(probability, 6),
            "predicted_defect_process": target_process,
            "risk_grade": probability_to_risk_grade(probability),
            "main_cause": feature_names[top_feature_idx],
            "source_process_influence_score": round(process_scores.get(source_process, 0.0), 6),
            "label_source": metadata_row["label_source"],
            "predicted_at": predicted_at,
        })

    return pd.DataFrame(records).sort_values(
        ["target_defect_probability", "source_process_influence_score"],
        ascending=False,
    )


transfer_predictions = build_transfer_predictions(
    meta_shap,
    proba_shap,
    shap_matrix,
    X_shap.columns.tolist(),
    feature_process,
)
display(transfer_predictions.head(30))

risk_summary = transfer_predictions.groupby(
    ["source_process_code", "target_process_code", "risk_grade"]
).agg(
    vehicle_count=("car_master_id", "nunique"),
    prediction_count=("sample_event_id", "count"),
    actual_defect_ratio=("target_defect_label", "mean"),
    avg_target_defect_probability=("target_defect_probability", "mean"),
    avg_source_process_influence=("source_process_influence_score", "mean"),
).reset_index().sort_values(
    ["avg_target_defect_probability", "prediction_count"],
    ascending=False,
)
display(risk_summary)


In [ ]:
# 전이 위험 시각화
plt.figure(figsize=(9, 5))
top_flow = risk_summary.head(12).copy()
top_flow["flow"] = top_flow["source_process_code"] + " -> " + top_flow["target_process_code"] + " (" + top_flow["risk_grade"] + ")"
sns.barplot(data=top_flow, y="flow", x="avg_target_defect_probability", hue="risk_grade", dodge=False)
plt.title("Process Transfer Risk by SHAP Influence")
plt.xlabel("Avg Final Defect Probability")
plt.ylabel("")
plt.legend(title="Risk")
plt.tight_layout()
plt.show()

# 불량 확률 분포 시각화
plt.figure(figsize=(8, 4))
sns.histplot(data=transfer_predictions, x="target_defect_probability", hue="risk_grade", bins=40, multiple="stack")
plt.title("Predicted Final Defect Probability Distribution")
plt.xlabel("Defect Probability")
plt.tight_layout()
plt.show()


## 12. AI 모델 관리 항목

모델 선정 근거, 데이터셋 규모, 테스트 케이스 수, Accuracy, False Positive Rate, 성능 개선 현황을 운영 관리용 요약으로 정리합니다.


In [ ]:
ai_management_table = pd.DataFrame([
    {"관리 항목": "모델 선정 근거 정리", "값": ai_management_summary["모델_선정_근거"]},
    {"관리 항목": "데이터셋 규모 관리", "값": json.dumps(ai_management_summary["데이터셋_규모_관리"], ensure_ascii=False)},
    {"관리 항목": "테스트 케이스 수 관리", "값": json.dumps(ai_management_summary["테스트_케이스_수_관리"], ensure_ascii=False)},
    {"관리 항목": "정확도(Accuracy) 측정", "값": round(ai_management_summary["정확도_Accuracy"], 6)},
    {"관리 항목": "오탐률(False Positive Rate) 측정", "값": round(ai_management_summary["오탐률_False_Positive_Rate"], 6)},
    {"관리 항목": "모델 성능 개선 현황 관리", "값": json.dumps(ai_management_summary["모델_성능_개선_현황"], ensure_ascii=False)},
])

display(ai_management_table)


## 13. 모델 및 결과 저장


In [ ]:
# 저장 경로
selected_model_path = OUTPUT_DIR / "selected_defect_detector.joblib"
selected_model_pkl_path = OUTPUT_DIR / "selected_defect_detector.pkl"
candidate_models_path = OUTPUT_DIR / "candidate_defect_models.joblib"
candidate_models_pkl_path = OUTPUT_DIR / "candidate_defect_models.pkl"
model_path = OUTPUT_DIR / "lightgbm_defect_detector.joblib"
model_pkl_path = OUTPUT_DIR / "lightgbm_defect_detector.pkl"
feature_path = OUTPUT_DIR / "defect_model_features.json"
metrics_path = OUTPUT_DIR / "defect_model_metrics.json"
comparison_path = OUTPUT_DIR / "defect_model_comparison.csv"
confusion_matrix_path = OUTPUT_DIR / "defect_model_confusion_matrices.json"
ai_management_path = OUTPUT_DIR / "defect_ai_management_summary.json"
importance_path = OUTPUT_DIR / "defect_model_feature_importance.csv"
all_importance_path = OUTPUT_DIR / "defect_all_model_feature_importance.csv"
shap_importance_path = OUTPUT_DIR / "defect_shap_importance.csv"
transfer_path = OUTPUT_DIR / "defect_transfer_prediction_result.csv"
risk_summary_path = OUTPUT_DIR / "defect_transfer_risk_summary.csv"

# 모델 저장
joblib.dump(model, selected_model_path)
joblib.dump(trained_models, candidate_models_path)
with open(selected_model_pkl_path, "wb") as f:
    pickle.dump(model, f)
with open(candidate_models_pkl_path, "wb") as f:
    pickle.dump(trained_models, f)

# 기존 LightGBM 파일명 호환 저장
if "LightGBM" in trained_models:
    joblib.dump(trained_models["LightGBM"], model_path)
    with open(model_pkl_path, "wb") as f:
        pickle.dump(trained_models["LightGBM"], f)

feature_path.write_text(json.dumps(feature_cols, ensure_ascii=False, indent=2), encoding="utf-8")

confusion_payload = {
    name: matrix.tolist()
    for name, matrix in model_confusion_matrices.items()
}
confusion_matrix_path.write_text(json.dumps(confusion_payload, ensure_ascii=False, indent=2), encoding="utf-8")
ai_management_path.write_text(json.dumps(ai_management_summary, ensure_ascii=False, indent=2), encoding="utf-8")

# 평가 지표 저장
metrics = {
    "selected_model_name": selected_model_name,
    "roc_auc": float(roc_auc),
    "average_precision": float(pr_auc),
    "accuracy": float(accuracy),
    "false_positive_rate": float(false_positive_rate),
    "best_threshold": float(best_threshold),
    "f1_at_best_threshold": float(f1_score(y_valid, valid_pred, zero_division=0)),
    "confusion_matrix": cm.tolist(),
    "train_rows": int(len(X_train)),
    "valid_rows": int(len(X_valid)),
    "feature_count": int(len(feature_cols)),
    "positive_ratio_train": float(y_train.mean()),
    "candidate_models": list(trained_models.keys()),
    "cv_splits": int(effective_cv_splits),
    "cv_trials_per_model": int(CV_N_ITER if ENABLE_HYPERPARAMETER_TUNING else 1),
    "lightgbm_tuning_method": "Optuna" if ENABLE_OPTUNA_TUNING else "ParameterSampler",
    "smote_enabled": bool(USE_SMOTE),
    "smote_sampling_strategy": float(SMOTE_SAMPLING_STRATEGY),
    "threshold_strategy": {
        "type": "recall_priority",
        "target_recall": float(RECALL_PRIORITY_TARGET),
        "min_precision": float(RECALL_PRIORITY_MIN_PRECISION),
        "beta": float(THRESHOLD_BETA),
    },
    "model_selection_reason": ai_management_summary["모델_선정_근거"],
    "data_source": "manufacturing_event_json.event_json",
    "label_restore_strategy": preprocessing_metadata["label_sources"],
    "transition_summary": transition_summary.to_dict(orient="records"),
    "created_at": datetime.now(timezone.utc).isoformat(),
}
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

# 분석 결과 저장
cv_results.to_csv(cv_results_path, index=False)
if "cv_fold_results" in globals():
    cv_fold_results.to_csv(cv_fold_results_path, index=False)
model_comparison.to_csv(comparison_path, index=False)
importance_df.to_csv(importance_path, index=False)
all_model_importance.to_csv(all_importance_path, index=False)
shap_importance.to_csv(shap_importance_path, index=False)
transfer_predictions.to_csv(transfer_path, index=False)
risk_summary.to_csv(risk_summary_path, index=False)

print("Saved outputs:")
for path in [
    selected_model_path,
    selected_model_pkl_path,
    candidate_models_path,
    candidate_models_pkl_path,
    model_path,
    model_pkl_path,
    feature_path,
    metrics_path,
    cv_results_path,
    comparison_path,
    confusion_matrix_path,
    ai_management_path,
    importance_path,
    all_importance_path,
    shap_importance_path,
    transfer_path,
    risk_summary_path,
]:
    print(path)
